# Book-level data and descriptive analysis

This publication notebook uses the corrected authoritative one-row-per-book analysis table for the descriptive corpus analyses used in the manuscript.

The manuscript output in this notebook is **Paper Figure 2** (mean book-level interiority across selected, overlapping genre indicators and historical periods). The corresponding eight-indicator book-count heatmap is exported as the Appendix D figure. Final-paper Figure 7 is produced in `12_canonical_author_historical_analysis.ipynb` and is not duplicated here.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/interiority_mplconfig")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".gitignore").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

ROOT = find_repository_root(Path.cwd().resolve())
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from analysis_utils import (
    BOOTSTRAP_REPETITIONS,
    BOOTSTRAP_SEED,
    BROAD_GROUP_COLORS,
    BROAD_GROUP_LABELS,
    BROAD_GROUP_MARKERS,
    BROAD_GROUP_ORDER,
    DOMESTIC_HYBRID_ORANGE,
    DOMESTIC_ORANGE,
    GENRE_COLUMNS,
    GENRE_LABELS,
    LIGHT_GRAY,
    NONDOMESTIC_BLUE,
    NONDOMESTIC_DARK,
    PERIOD_COLORS,
    PERIOD_MARKERS,
    PUBLICATION_PERIODS,
    RESIDUAL_GRAY,
    SUBSTANTIVE_GENRES,
    TIME_PERIOD_ORDER,
    bootstrap_difference_summary,
    bootstrap_mean_summary,
    configure_analysis_style,
    draw_period_estimate,
    rolling_book_bootstrap,
    save_figure,
)

ANALYSIS_TABLE_PATH = ROOT / "data" / "processed" / "book_summary_analysis.csv"
ANALYSIS_SCHEMA_PATH = ROOT / "data" / "processed" / "book_summary_analysis_schema.csv"
RESULTS_DIR = ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

configure_analysis_style()


## Corpus and descriptive analyses

Figure 2 and the Appendix D count figure use the corrected authoritative dataset at `data/processed/book_summary_analysis.csv`. The earlier publication-era snapshot remains at `data/frozen/publication_snapshot/book_summary_analysis_publication.csv` as historical provenance only; it is not the source for the current manuscript figures. Neither dataset is rebuilt or modified by this notebook.


In [ ]:
book_summary_analysis = pd.read_csv(ANALYSIS_TABLE_PATH, low_memory=False)
if book_summary_analysis["gutenberg_id"].duplicated().any():
    raise ValueError("Descriptive analyses require exactly one row per book.")
print(f"Loaded finalized analysis table: {ANALYSIS_TABLE_PATH}")
print(f"Rows: {len(book_summary_analysis):,}; distinct books: {book_summary_analysis['gutenberg_id'].nunique():,}")


### Overall interiority-score distribution

The distribution uses one base-sample score per book. Missing book scores are counted explicitly and excluded from numerical summaries and the histogram.


In [ ]:
score = book_summary_analysis["avg_interiority_score"].dropna()

overall_score_summary = pd.DataFrame(
    {
        "metric": [
            "all_books",
            "books_with_score",
            "books_missing_score",
            "mean",
            "standard_deviation",
            "minimum",
            "5th_percentile",
            "25th_percentile",
            "median",
            "75th_percentile",
            "95th_percentile",
            "maximum",
        ],
        "value": [
            len(book_summary_analysis),
            score.count(),
            book_summary_analysis["avg_interiority_score"].isna().sum(),
            score.mean(),
            score.std(),
            score.min(),
            score.quantile(0.05),
            score.quantile(0.25),
            score.median(),
            score.quantile(0.75),
            score.quantile(0.95),
            score.max(),
        ],
    }
)
display(overall_score_summary.style.format({"value": "{:.3f}"}))

fig, axes = plt.subplots(
    2,
    1,
    figsize=(9, 6),
    gridspec_kw={"height_ratios": [4, 1]},
    sharex=True,
)
sns.histplot(
    score,
    bins=24,
    kde=True,
    color="#3A6EA5",
    ax=axes[0],
)
axes[0].axvline(
    score.mean(),
    color="#D9773D",
    linewidth=1.8,
    label=f"Mean = {score.mean():.3f}",
)
axes[0].axvline(
    score.median(),
    color="#6A994E",
    linewidth=1.8,
    label=f"Median = {score.median():.3f}",
)
axes[0].set_title("Distribution of book-level average interiority scores")
axes[0].set_ylabel("Books")
axes[0].legend(frameon=False)

sns.boxplot(
    x=score,
    color="#A9C5DF",
    width=0.45,
    ax=axes[1],
)
axes[1].set_xlabel("Average interiority score")
axes[1].set_ylabel("")
for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()


### Highest and lowest five books


In [ ]:
ranked_books = (
    book_summary_analysis.dropna(subset=["avg_interiority_score"])
    .sort_values(
        ["avg_interiority_score", "title", "gutenberg_id"],
        ascending=[False, True, True],
    )
)
book_rank_columns = [
    "gutenberg_id",
    "title",
    "author_display",
    "publication_year",
    "broad_group",
    "avg_interiority_score",
    "genre_tags"
]
highest_five_books = ranked_books[book_rank_columns].head(5)
lowest_five_books = (
    ranked_books.sort_values(
        ["avg_interiority_score", "title", "gutenberg_id"],
        ascending=[True, True, True],
    )[book_rank_columns]
    .head(5)
)

display(Markdown("**Highest five books**"))
display(highest_five_books.style.format({"avg_interiority_score": "{:.3f}"}))
display(Markdown("**Lowest five books**"))
display(lowest_five_books.style.format({"avg_interiority_score": "{:.3f}"}))


### Broad-group summary


In [ ]:
broad_group_overall_summary = (
    book_summary_analysis.assign(
        broad_group_summary=book_summary_analysis["broad_group"].fillna("missing")
    )
    .groupby("broad_group_summary", as_index=False)
    .agg(
        books=("gutenberg_id", "nunique"),
        books_with_score=("avg_interiority_score", "count"),
        average_interiority=("avg_interiority_score", "mean"),
    )
)
broad_group_overall_summary["book_share"] = (
    broad_group_overall_summary["books"]
    / broad_group_overall_summary["books"].sum()
)

display(
    broad_group_overall_summary.sort_values(
        "books", ascending=False
    ).style.format(
        {
            "average_interiority": "{:.3f}",
            "book_share": "{:.1%}",
        }
    )
)


In [ ]:
broad_group_boxplot_data = (
    book_summary_analysis.assign(
        broad_group_plot=book_summary_analysis["broad_group"].fillna("missing")
    )
    .dropna(subset=["avg_interiority_score"])
)
broad_group_box_order = (
    broad_group_overall_summary.sort_values(
        ["average_interiority", "broad_group_summary"],
        ascending=[False, True],
        na_position="last",
    )["broad_group_summary"].tolist()
)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.boxplot(
    data=broad_group_boxplot_data,
    x="avg_interiority_score",
    y="broad_group_plot",
    hue="broad_group_plot",
    order=broad_group_box_order,
    hue_order=broad_group_box_order,
    palette=BROAD_GROUP_COLORS,
    dodge=False,
    legend=False,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "#3F464D",
        "markersize": 5.5,
    },
    ax=ax,
)
ax.set_title("Book-level interiority distributions by broad group")
ax.set_xlabel("Average interiority score")
ax.set_ylabel("Broad group")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


### Overlapping indicator-group summary


In [ ]:
SUMMARY_INDICATOR_COLUMNS = [
    "is_domestic",
    "is_detective_mystery",
    "is_science_fiction",
    "is_adventure",
    "is_gothic",
    "is_historical",
    "is_canonical_author",
    "is_english_fiction_19c",
]
SUMMARY_INDICATOR_LABELS = {
    "is_domestic": "Domestic",
    "is_detective_mystery": "Detective / mystery",
    "is_science_fiction": "Science fiction",
    "is_adventure": "Adventure",
    "is_gothic": "Gothic",
    "is_historical": "Historical",
    "is_canonical_author": "Canonical author",
    "is_english_fiction_19c": "English fiction, 19th c.",
}

indicator_overall_rows = []
for group_column in SUMMARY_INDICATOR_COLUMNS:
    group_books = (
        book_summary_analysis[
            ["gutenberg_id", "avg_interiority_score", group_column]
        ]
        .dropna(subset=[group_column])
        .loc[lambda frame: frame[group_column].eq(1)]
    )
    indicator_overall_rows.append(
        {
            "group_column": group_column,
            "group_label": SUMMARY_INDICATOR_LABELS[group_column],
            "books": group_books["gutenberg_id"].nunique(),
            "books_with_score": group_books["avg_interiority_score"].count(),
            "average_interiority": group_books["avg_interiority_score"].mean(),
        }
    )

indicator_overall_summary = pd.DataFrame(indicator_overall_rows)
display(
    indicator_overall_summary.style.format(
        {"average_interiority": "{:.3f}"}
    )
)


In [ ]:
indicator_boxplot_frames = []
for group_column in SUMMARY_INDICATOR_COLUMNS:
    group_scores = (
        book_summary_analysis[
            ["gutenberg_id", "avg_interiority_score", group_column]
        ]
        .dropna(subset=["avg_interiority_score", group_column])
        .loc[lambda frame: frame[group_column].eq(1)]
        .copy()
    )
    group_scores["group_label"] = SUMMARY_INDICATOR_LABELS[group_column]
    indicator_boxplot_frames.append(group_scores)

indicator_boxplot_data = pd.concat(
    indicator_boxplot_frames,
    ignore_index=True,
)
indicator_box_order = (
    indicator_overall_summary.sort_values(
        ["average_interiority", "group_label"],
        ascending=[False, True],
        na_position="last",
    )["group_label"].tolist()
)

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.boxplot(
    data=indicator_boxplot_data,
    x="avg_interiority_score",
    y="group_label",
    order=indicator_box_order,
    color="#B8D5B0",
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "#D9773D",
        "markeredgecolor": "white",
        "markersize": 6,
    },
    ax=ax,
)
ax.set_title("Book-level interiority distributions by indicator group")
ax.set_xlabel("Average interiority score")
ax.set_ylabel("")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


The eight indicator groups overlap, so their book counts should not be added together. All subgroup means are equal-book averages and automatically exclude missing book-level scores.


### Narrative-mode-base summary

`narrative_mode_base` consolidates candidate and reviewed narrative-mode labels into three broader descriptive categories.


In [ ]:
NARRATIVE_MODE_BASE_MAP = {
    "third_person": "third_person",
    "third_person_candidate": "third_person",
    "epistolary": "epistolary",
    "epistolary_candidate": "epistolary",
    "first_person": "first_person_or_mixed",
    "first_person_candidate": "first_person_or_mixed",
    "mixed": "first_person_or_mixed",
}
book_summary_analysis["narrative_mode_base"] = (
    book_summary_analysis["narrative_mode"].replace(NARRATIVE_MODE_BASE_MAP)
)

narrative_mode_base_summary = (
    book_summary_analysis.assign(
        narrative_mode_base_summary=book_summary_analysis[
            "narrative_mode_base"
        ].fillna("missing")
    )
    .groupby("narrative_mode_base_summary", as_index=False)
    .agg(
        books=("gutenberg_id", "nunique"),
        books_with_score=("avg_interiority_score", "count"),
        average_interiority=("avg_interiority_score", "mean"),
    )
)
narrative_mode_base_summary["book_share"] = (
    narrative_mode_base_summary["books"]
    / narrative_mode_base_summary["books"].sum()
)

display(
    narrative_mode_base_summary.sort_values(
        ["books", "narrative_mode_base_summary"],
        ascending=[False, True],
    ).style.format(
        {
            "average_interiority": "{:.3f}",
            "book_share": "{:.1%}",
        }
    )
)


In [ ]:
narrative_mode_base_plot_data = (
    book_summary_analysis.assign(
        narrative_mode_base_plot=book_summary_analysis[
            "narrative_mode_base"
        ].fillna("missing")
    )
    .dropna(subset=["avg_interiority_score"])
)

narrative_mode_bootstrap_rows = []
for mode, mode_data in narrative_mode_base_plot_data.groupby(
    "narrative_mode_base_plot",
    sort=False,
):
    summary = bootstrap_mean_summary(
        mode_data["avg_interiority_score"],
        "narrative_mode_base",
        mode,
    )
    narrative_mode_bootstrap_rows.append(
        {
            "narrative_mode_base": mode,
            **summary,
        }
    )

narrative_mode_base_bootstrap = (
    pd.DataFrame(narrative_mode_bootstrap_rows)
    .sort_values(
        ["estimate", "narrative_mode_base"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)
narrative_mode_base_order = narrative_mode_base_bootstrap[
    "narrative_mode_base"
].tolist()
narrative_mode_base_palette = dict(
    zip(
        narrative_mode_base_order,
        sns.color_palette(
            "crest",
            n_colors=len(narrative_mode_base_order),
        ),
    )
)
if "missing" in narrative_mode_base_palette:
    narrative_mode_base_palette["missing"] = "#D9DDE1"

NARRATIVE_MODE_BASE_LABELS = {
    "epistolary": "Epistolary",
    "first_person_or_mixed": "First person or mixed",
    "third_person": "Third person",
    "missing": "Missing",
}

fig, ax = plt.subplots(figsize=(7, 4.5))
for position, row in narrative_mode_base_bootstrap.iterrows():
    mode = row["narrative_mode_base"]
    estimate = row["estimate"]
    ci_low = row["ci_low"]
    ci_high = row["ci_high"]
    color = narrative_mode_base_palette[mode]
    ax.errorbar(
        position,
        estimate,
        yerr=[[estimate - ci_low], [ci_high - estimate]],
        fmt="o",
        markersize=8,
        color=color,
        markeredgecolor="white",
        markeredgewidth=0.7,
        ecolor=color,
        elinewidth=1.8,
        capsize=4,
        capthick=1.5,
        zorder=3,
    )
    ax.text(
        position,
        ci_high + 0.018,
        f"n={int(row['n_books']):,}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color="#555B61",
    )

ax.set_xticks(
    range(len(narrative_mode_base_order)),
    [
        NARRATIVE_MODE_BASE_LABELS.get(mode, mode)
        for mode in narrative_mode_base_order
    ],
)
ax.set_title("Mean book-level interiority by narrative mode")
ax.set_xlabel("Narrative mode")
ax.set_ylabel("Average interiority score")
ax.grid(axis="x", visible=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(
    Markdown(
        "Points are equal-book means; whiskers are 95% book-level bootstrap "
        f"CIs based on **{BOOTSTRAP_REPETITIONS:,}** resamples with a fixed seed."
    )
)


## Paper Figure 2 - Indicator-group average interiority by historical period

The genre indicators overlap and are not made mutually exclusive. Paper Figure 2 reports equal-book average interiority for the six selected manuscript genre groups. The corresponding book counts for all eight tracked indicators are exported as a separate appendix figure.


In [ ]:
# Reuse the manuscript's exact eight overlapping indicator groups.
LINE_GROUP_COLUMNS = GENRE_COLUMNS
LINE_LABELS = GENRE_LABELS

indicator_time_frames = []

for group_column in LINE_GROUP_COLUMNS:
    group_books = (
        book_summary_analysis[
            [
                "gutenberg_id",
                "time_period",
                "avg_interiority_score",
                group_column,
            ]
        ]
        .dropna(subset=["time_period", group_column])
        .loc[lambda frame: frame[group_column].eq(1)]
        .copy()
    )

    group_summary = (
        group_books.groupby(
            "time_period",
            observed=True,
            as_index=False,
        )
        .agg(
            books=("gutenberg_id", "nunique"),
            books_with_score=("avg_interiority_score", "count"),
            average_interiority=("avg_interiority_score", "mean"),
        )
    )
    group_summary["group_column"] = group_column
    group_summary["group_label"] = LINE_LABELS[group_column]
    indicator_time_frames.append(group_summary)

indicator_time_summary = pd.concat(
    indicator_time_frames,
    ignore_index=True,
)

display(
    indicator_time_summary[
        [
            "time_period",
            "group_column",
            "books",
            "books_with_score",
            "average_interiority",
        ]
    ].style.format({"average_interiority": "{:.3f}"})
)


In [ ]:
indicator_label_order = [
    LINE_LABELS[column] for column in LINE_GROUP_COLUMNS
]

indicator_count_matrix = (
    indicator_time_summary.pivot(
        index="group_label",
        columns="time_period",
        values="books",
    )
    .reindex(
        index=indicator_label_order,
        columns=TIME_PERIOD_ORDER,
    )
)
indicator_score_matrix = (
    indicator_time_summary.pivot(
        index="group_label",
        columns="time_period",
        values="average_interiority",
    )
    .reindex(
        index=indicator_label_order,
        columns=TIME_PERIOD_ORDER,
    )
)
score_row_order = (
    indicator_score_matrix["1900+"]
    .sort_values(ascending=False, na_position="last")
    .index
)
figure_2_excluded_labels = {
    LINE_LABELS["is_canonical_author"],
    LINE_LABELS["is_english_fiction_19c"],
}
score_row_order = [
    label for label in score_row_order
    if label not in figure_2_excluded_labels
]
indicator_score_matrix = indicator_score_matrix.reindex(index=score_row_order)

count_row_order = (
    indicator_count_matrix["1900+"]
    .sort_values(ascending=False, na_position="last")
    .index
)
indicator_count_display = (
    indicator_count_matrix
    .reindex(index=count_row_order)
    .fillna(0)
)
indicator_count_vmin = 0
indicator_count_vmax = np.nanmax(indicator_count_matrix.to_numpy())
indicator_score_annotations = indicator_score_matrix.apply(
    lambda column: column.map(
        lambda value: "—" if pd.isna(value) else f"{value:.2f}"
    )
)
period_tick_labels = ["Pre-1840", "1840–1899", "1900+"]

from matplotlib.colors import LinearSegmentedColormap

score_values = indicator_score_matrix.to_numpy(dtype=float)
score_vmin = np.nanmin(score_values)
score_vmax = np.nanmax(score_values)
score_cmap = LinearSegmentedColormap.from_list(
    "morandi",
    ["#F8F5F7", "#B8A3B1"],
)

fig, ax = plt.subplots(figsize=(4.8, 3.0))
sns.heatmap(
    indicator_score_matrix,
    mask=indicator_score_matrix.isna(),
    annot=False,
    cmap=score_cmap,
    vmin=score_vmin,
    vmax=score_vmax,
    linewidths=1.8,
    linecolor="white",
    cbar=False,
    ax=ax,
)
ax.set_facecolor("#F7F6F4")
ax.grid(False)
for row_index in range(indicator_score_matrix.shape[0]):
    for column_index in range(indicator_score_matrix.shape[1]):
        value = indicator_score_matrix.iloc[row_index, column_index]
        if pd.isna(value):
            label = "—"
        else:
            label = f"{value:.2f}"
        ax.text(
            column_index + 0.5,
            row_index + 0.5,
            label,
            ha="center",
            va="center",
            fontsize=9,
            color="#3D3D3A",
            fontweight="normal",
        )
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(period_tick_labels, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.tick_params(
    axis="both",
    which="both",
    length=0,
    labelsize=9,
    colors="#3D3D3A",
)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout(pad=0.5)
figure_2_table_dir = TABLE_DIR
figure_2_table_dir.mkdir(parents=True, exist_ok=True)
indicator_time_summary.to_csv(
    figure_2_table_dir / "figure_2_indicator_group_period_summary.csv",
    index=False,
)
save_figure(
    fig,
    FIGURE_DIR,
    "figure_2_indicator_group_heatmaps",
)
plt.show()

count_cmap = sns.light_palette("#8EA8C6", as_cmap=True)
count_fig, count_ax = plt.subplots(figsize=(4.8, 3.8))
sns.heatmap(
    indicator_count_display,
    annot=False,
    cmap=count_cmap,
    vmin=indicator_count_vmin,
    vmax=indicator_count_vmax,
    linewidths=1.8,
    linecolor="white",
    cbar=False,
    ax=count_ax,
)
count_ax.grid(False)
for row_index in range(indicator_count_display.shape[0]):
    for column_index in range(indicator_count_display.shape[1]):
        value = indicator_count_display.iloc[row_index, column_index]
        count_ax.text(
            column_index + 0.5,
            row_index + 0.5,
            f"{value:.0f}",
            ha="center",
            va="center",
            fontsize=9,
            color="#3D3D3A",
            fontweight="normal",
        )
count_ax.set_xlabel("")
count_ax.set_ylabel("")
count_ax.set_xticklabels(period_tick_labels, rotation=0)
count_ax.set_yticklabels(count_ax.get_yticklabels(), rotation=0)
count_ax.tick_params(
    axis="both",
    which="both",
    length=0,
    labelsize=9,
    colors="#3D3D3A",
)
for spine in count_ax.spines.values():
    spine.set_visible(False)
plt.tight_layout(pad=0.5)
save_figure(
    count_fig,
    FIGURE_DIR,
    "appendix_figure_2_book_counts",
)
plt.show()


## Supporting descriptive analyses

The following retained summaries and figures are useful corpus checks, robustness views, or exploratory descriptions; they are not numbered manuscript figures.


### Interiority time series by book classification

The plotting workflow below is reusable across time columns. For each indicator, it keeps books where the indicator equals `1`, drops rows with missing indicator, time, or average interiority score, and never imputes missing values.


In [ ]:
LINE_GROUP_COLUMNS = [
    "is_domestic",
    "is_detective_mystery",
    "is_science_fiction",
    "is_adventure",
    "is_gothic",
    "is_historical",
    "is_canonical_author",
    "is_english_fiction_19c",
]

LINE_LABELS = {
    "is_domestic": "Domestic",
    "is_detective_mystery": "Detective / mystery",
    "is_science_fiction": "Science fiction",
    "is_adventure": "Adventure",
    "is_gothic": "Gothic",
    "is_historical": "Historical",
    "is_canonical_author": "Canonical author",
    "is_english_fiction_19c": "English fiction, 19th c.",
}

LINE_COLORS = dict(
    zip(
        LINE_GROUP_COLUMNS,
        sns.color_palette("tab10", n_colors=len(LINE_GROUP_COLUMNS)),
    )
)

In [ ]:
def add_year_period_column(
    data,
    width,
    year_column="publication_year",
    output_column=None,
):
    """Return a copy with an ordered year-period column such as 1700-1724."""
    if width <= 0:
        raise ValueError("width must be a positive integer")
    if year_column not in data.columns:
        raise KeyError(f"Time source column not found: {year_column}")

    output_column = output_column or f"period_{width}yr"
    result = data.copy()
    years = pd.to_numeric(result[year_column], errors="coerce")
    valid = years.notna()

    period_labels = pd.Series(pd.NA, index=result.index, dtype="string")
    starts = (years.loc[valid].astype(int) // width) * width
    period_labels.loc[valid] = (
        starts.astype(str) + "-" + (starts + width - 1).astype(str)
    )

    if valid.any():
        first_start = int(starts.min())
        last_start = int(starts.max())
        ordered_labels = [
            f"{start}-{start + width - 1}"
            for start in range(first_start, last_start + width, width)
        ]
    else:
        ordered_labels = []

    result[output_column] = pd.Categorical(
        period_labels,
        categories=ordered_labels,
        ordered=True,
    )
    return result


book_summary_analysis = add_year_period_column(
    book_summary_analysis,
    width=25,
    output_column="period_25yr",
)
book_summary_analysis = add_year_period_column(
    book_summary_analysis,
    width=50,
    output_column="period_50yr",
)

display(
    book_summary_analysis[
        ["publication_year", "period_25yr", "period_50yr"]
    ].drop_duplicates().sort_values("publication_year").head(12)
)

In [ ]:
def plot_indicator_timeseries(
    data,
    time_column,
    group_columns=LINE_GROUP_COLUMNS,
    score_column="avg_interiority_score",
    title=None,
    xlabel=None,
    figsize=(12, 7),
):
    """Plot one equal-book mean line per binary indicator for any time column."""
    required = ["gutenberg_id", time_column, score_column, *group_columns]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(f"Required columns not found: {missing}")

    line_frames = []
    coverage_rows = []

    for group_column in group_columns:
        group_books = (
            data[
                [
                    "gutenberg_id",
                    time_column,
                    score_column,
                    group_column,
                ]
            ]
            .dropna(subset=[time_column, score_column, group_column])
            .loc[lambda frame: frame[group_column].eq(1)]
            .copy()
        )

        grouped = (
            group_books.groupby(time_column, observed=True, as_index=False)
            .agg(
                avg_interiority_score=(score_column, "mean"),
                books_in_period=("gutenberg_id", "nunique"),
            )
        )
        grouped["group_column"] = group_column
        line_frames.append(grouped)

        coverage_rows.append(
            {
                "group_column": group_column,
                "books": group_books["gutenberg_id"].nunique(),
                "time_values": grouped[time_column].nunique(),
                "first_time_value": (
                    grouped[time_column].min() if len(grouped) else pd.NA
                ),
                "last_time_value": (
                    grouped[time_column].max() if len(grouped) else pd.NA
                ),
            }
        )

    line_data = pd.concat(line_frames, ignore_index=True)
    coverage = pd.DataFrame(coverage_rows)

    is_categorical = isinstance(data[time_column].dtype, pd.CategoricalDtype)
    if is_categorical:
        observed_values = set(line_data[time_column].dropna())
        time_order = [
            value
            for value in data[time_column].cat.categories
            if value in observed_values
        ]
        time_positions = {
            value: position for position, value in enumerate(time_order)
        }

    fig, ax = plt.subplots(figsize=figsize)
    for group_column in group_columns:
        series = line_data.loc[
            line_data["group_column"].eq(group_column)
        ].copy()

        if is_categorical:
            series["_plot_x"] = series[time_column].map(time_positions).astype(float)
            series = series.sort_values("_plot_x")
            x_values = series["_plot_x"]
            marker = "o"
        else:
            series = series.sort_values(time_column)
            x_values = series[time_column]
            marker = None

        ax.plot(
            x_values,
            series["avg_interiority_score"],
            color=LINE_COLORS.get(group_column),
            linewidth=1.8,
            marker=marker,
            markersize=4.5,
            alpha=0.88,
            label=LINE_LABELS.get(group_column, group_column),
        )

    if is_categorical:
        ax.set_xticks(range(len(time_order)), time_order, rotation=45, ha="right")

    ax.set_xlabel(xlabel or time_column)
    ax.set_ylabel("Average interiority score")
    ax.set_title(title or f"Average interiority by {time_column}")
    ax.legend(
        title="Books with indicator = 1",
        frameon=False,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

    display(Markdown(f"**Coverage for `{time_column}` after dropping NA**"))
    display(coverage)
    return line_data, coverage, ax

#### Annual series

In [ ]:
annual_line_data, annual_line_coverage, annual_line_ax = (
    plot_indicator_timeseries(
        book_summary_analysis,
        time_column="publication_year",
        title="Annual average interiority by book classification",
        xlabel="Publication year",
    )
)

#### 25-year series

In [ ]:
period_25yr_line_data, period_25yr_coverage, period_25yr_ax = (
    plot_indicator_timeseries(
        book_summary_analysis,
        time_column="period_25yr",
        title="Average interiority by 25-year period and book classification",
        xlabel="25-year publication period",
    )
)

#### 50-year series

In [ ]:
period_50yr_line_data, period_50yr_coverage, period_50yr_ax = (
    plot_indicator_timeseries(
        book_summary_analysis,
        time_column="period_50yr",
        title="Average interiority by 50-year period and book classification",
        xlabel="50-year publication period",
    )
)

All three charts use the same function. To use another time variable later, create or select that column and pass its name through `time_column`. No missing values are imputed.

### Distribution across three historical periods

The finalized `time_period` values are `<1840`, `1840-1899`, and `1900+`. Counts include every book in the relevant group; average interiority uses only books with non-missing `avg_interiority_score`.


In [ ]:
time_period_summary = (
    book_summary_analysis.groupby(
        "time_period",
        observed=True,
        as_index=False,
    )
    .agg(
        books=("gutenberg_id", "nunique"),
        books_with_score=("avg_interiority_score", "count"),
        average_interiority=("avg_interiority_score", "mean"),
        median_interiority=("avg_interiority_score", "median"),
        std_interiority=("avg_interiority_score", "std"),
        minimum_interiority=("avg_interiority_score", "min"),
        maximum_interiority=("avg_interiority_score", "max"),
    )
)

display(
    time_period_summary.style.format(
        {
            "average_interiority": "{:.3f}",
            "median_interiority": "{:.3f}",
            "std_interiority": "{:.3f}",
            "minimum_interiority": "{:.3f}",
            "maximum_interiority": "{:.3f}",
        }
    )
)


#### Distribution by `broad_group_analysis`


In [ ]:
broad_group_analysis_for_summary = book_summary_analysis.assign(
    
    
    
    _plot=book_summary_analysis["broad_group_analysis"].fillna("missing")
)

broad_group_analysis_time_summary = (
    broad_group_analysis_for_summary.groupby(
        ["time_period", "broad_group_analysis"],
        observed=True,
        as_index=False,
    )
    .agg(
        books=("gutenberg_id", "nunique"),
        books_with_score=("avg_interiority_score", "count"),
        average_interiority=("avg_interiority_score", "mean"),
    )
)
broad_group_analysis_time_summary["share_within_time_period"] = (
    broad_group_analysis_time_summary["books"]
    / broad_group_analysis_time_summary.groupby("time_period", observed=True)["books"].transform("sum")
)

display(
    broad_group_analysis_time_summary.style.format(
        {
            "average_interiority": "{:.3f}",
            "share_within_time_period": "{:.1%}",
        }
    )
)


In [ ]:
broad_group_analysis_order = [
    group
    for group in [
        "all_domestic",
        "major_genre_non_domestic",
        "other_non_domestic_non_major_genre",
        "missing",
    ]
    if group in set(broad_group_analysis_time_summary["broad_group_analysis"])
]

broad_group_analysis_colors = {
    "all_domestic": BROAD_GROUP_COLORS["domestic_only"],
    "major_genre_non_domestic": BROAD_GROUP_COLORS["major_genre_non_domestic"],
    "other_non_domestic_non_major_genre": BROAD_GROUP_COLORS[
        "other_non_domestic_non_major_genre"
    ],
    "missing": BROAD_GROUP_COLORS["missing"],
}

broad_count_matrix = (
    broad_group_analysis_time_summary.pivot(
        index="time_period",
        columns="broad_group_analysis",
        values="books",
    )
    .reindex(index=TIME_PERIOD_ORDER, columns=broad_group_analysis_order)
    .fillna(0)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
broad_count_matrix.plot(
    kind="bar",
    stacked=True,
    color=[
        broad_group_analysis_colors.get(group, "#6C757D")
        for group in broad_group_analysis_order
    ],
    ax=axes[0],
)
axes[0].set_title("Book counts by period and broad-group analysis")
axes[0].set_xlabel("Time period")
axes[0].set_ylabel("Number of books")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Broad-group analysis", frameon=False, fontsize=8)

sns.barplot(
    data=broad_group_analysis_time_summary,
    x="time_period",
    y="average_interiority",
    hue="broad_group_analysis",
    order=TIME_PERIOD_ORDER,
    hue_order=broad_group_analysis_order,
    palette={
        group: broad_group_analysis_colors.get(group, "#6C757D")
        for group in broad_group_analysis_order
    },
    errorbar=None,
    ax=axes[1],
)
axes[1].set_title("Average interiority by period and broad-group analysis")
axes[1].set_xlabel("Time period")
axes[1].set_ylabel("Average interiority score")
axes[1].legend(title="Broad-group analysis", frameon=False, fontsize=8)

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()


#### Indicator-group distribution table


The `broad_group` categories are mutually exclusive, while the eight Figure 2 indicator groups overlap. Consequently, indicator-group counts should not be summed to obtain the number of unique books in a period.


### Quotation metrics


In [ ]:
df_quote = book_summary_analysis.copy()

count_cols = [
    "n_narration_only",
    "n_narration_dominant",
    "n_quotation_dominant",
    "n_quotation_only"
]

metric_cols = [
    "overall_quote_share",
    "narration_quotation_interiority_gap",
    "narration_quotation_high_rate_gap",
    "high_narration_enrichment"
]

for col in count_cols + metric_cols:
    df_quote[col] = pd.to_numeric(df_quote[col], errors="coerce")

# Whether a book has enough passages on both sides
df_quote["gap_eligible"] = (
    df_quote["n_narration_dominant"].ge(10)
    & df_quote["n_quotation_dominant"].ge(10)
)

df_quote["min_dominant_n"] = df_quote[
    ["n_narration_dominant", "n_quotation_dominant"]
].min(axis=1)

print(f"Total books: {len(df_quote):,}")
print(
    f"Books eligible for gap metrics: "
    f"{df_quote['gap_eligible'].sum():,} "
    f"({df_quote['gap_eligible'].mean():.1%})"
)

In [ ]:
distribution_summary = (
    df_quote[metric_cols + count_cols]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
    .T
)

distribution_summary["missing_n"] = (
    df_quote[metric_cols + count_cols].isna().sum()
)

distribution_summary["missing_pct"] = (
    df_quote[metric_cols + count_cols].isna().mean()
)

display(distribution_summary.round(3))

Across the corpus, interiority is more often concentrated in narration-dominant than quotation-dominant passages. This pattern is especially consistent for the probability of receiving a HIGH interiority label, although substantial variation remains across individual books.

In [ ]:
quote_share_summary = pd.Series({
    "n_books": df_quote["overall_quote_share"].notna().sum(),
    "mean": df_quote["overall_quote_share"].mean(),
    "median": df_quote["overall_quote_share"].median(),
    "purely_nonquoted_pct": (
        df_quote["overall_quote_share"].eq(0).mean()
    ),
    "below_10pct_quote": (
        df_quote["overall_quote_share"].lt(0.10).mean()
    ),
    "above_50pct_quote": (
        df_quote["overall_quote_share"].gt(0.50).mean()
    ),
    "above_75pct_quote": (
        df_quote["overall_quote_share"].gt(0.75).mean()
    )
})

plt.figure(figsize=(8, 5))

plt.hist(
    df_quote["overall_quote_share"].dropna(),
    bins=np.linspace(0, 1, 21),
    edgecolor="white"
)

plt.axvline(
    df_quote["overall_quote_share"].median(),
    linestyle="--",
    label=f"Median = {df_quote['overall_quote_share'].median():.2f}"
)

plt.xlabel("Overall quote share")
plt.ylabel("Number of books")
plt.title("Distribution of Overall Quote Share")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
gap_metrics = [
    "narration_quotation_interiority_gap",
    "narration_quotation_high_rate_gap",
    "high_narration_enrichment"
]

near_zero_threshold = 0.05

sign_rows = []

for metric in gap_metrics:
    values = df_quote[metric].dropna()

    sign_rows.append({
        "metric": metric,
        "n_books": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "positive_pct": (values > near_zero_threshold).mean(),
        "near_zero_pct": (
            values.abs() <= near_zero_threshold
        ).mean(),
        "negative_pct": (values < -near_zero_threshold).mean()
    })

sign_summary = pd.DataFrame(sign_rows)

display(sign_summary.round(3))

In [ ]:
metric_titles = {
    "narration_quotation_interiority_gap":
        "Narration–Quotation Interiority Gap",

    "narration_quotation_high_rate_gap":
        "Narration–Quotation HIGH-Rate Gap",

    "high_narration_enrichment":
        "HIGH Narration Enrichment"
}

for metric in gap_metrics:
    values = df_quote[metric].dropna()

    plt.figure(figsize=(8, 5))

    plt.hist(
        values,
        bins=30,
        edgecolor="white"
    )

    plt.axvline(0, linewidth=1)
    plt.axvline(
        values.median(),
        linestyle="--",
        label=f"Median = {values.median():.3f}"
    )

    plt.xlabel(metric_titles[metric])
    plt.ylabel("Number of books")
    plt.title(f"Distribution of {metric_titles[metric]}")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
id_candidates = [
    "gutenberg_id",
    "book",
    "title",
    "author_display",
    "published_year",
    "category",
    "narrative_mode"
]

id_cols = [
    col for col in id_candidates
    if col in df_quote.columns
]

for metric in gap_metrics:
    display_cols = (
        id_cols
        + [
            "overall_quote_share",
            "n_narration_dominant",
            "n_quotation_dominant",
            metric
        ]
    )

    print(f"\nHighest values: {metric}")
    display(
        df_quote[display_cols]
        .dropna(subset=[metric])
        .nlargest(5, metric)
        .round(3)
    )

    print(f"\nLowest values: {metric}")
    display(
        df_quote[display_cols]
        .dropna(subset=[metric])
        .nsmallest(5, metric)
        .round(3)
    )

#### Author-level narration-quotation interiority gap

Each author's gap is the equal-book mean of the finalized book-level `narration_quotation_interiority_gap`. The ranking is restricted to authors represented by at least five books in the current dataset.


In [ ]:
AUTHOR_QUOTE_GAP_METRIC = "narration_quotation_interiority_gap"
MINIMUM_AUTHOR_BOOKS_FOR_QUOTE_RANKING = 5


def representative_author_display(values):
    observed = values.dropna().astype(str)
    if observed.empty:
        return pd.NA
    return observed.mode().iloc[0]


author_quote_gap_summary = (
    df_quote.drop_duplicates("gutenberg_id")
    .dropna(subset=["author_query"])
    .groupby("author_query", as_index=False)
    .agg(
        author_display=(
            "author_display",
            representative_author_display,
        ),
        n_books=("gutenberg_id", "nunique"),
        n_books_with_gap=(AUTHOR_QUOTE_GAP_METRIC, "count"),
        mean_narration_quotation_interiority_gap=(
            AUTHOR_QUOTE_GAP_METRIC,
            "mean",
        ),
    )
)
author_quote_gap_summary["gap_book_coverage"] = (
    author_quote_gap_summary["n_books_with_gap"]
    / author_quote_gap_summary["n_books"]
)
author_quote_gap_summary = author_quote_gap_summary.sort_values(
    [
        "mean_narration_quotation_interiority_gap",
        "n_books_with_gap",
        "author_display",
    ],
    ascending=[True, False, True],
    na_position="last",
).reset_index(drop=True)

author_quote_gap_top10_min5_books = (
    author_quote_gap_summary.loc[
        author_quote_gap_summary["n_books"].ge(
            MINIMUM_AUTHOR_BOOKS_FOR_QUOTE_RANKING
        )
        & author_quote_gap_summary[
            "mean_narration_quotation_interiority_gap"
        ].notna()
    ]
    .head(70)
    .reset_index(drop=True)
)
author_quote_gap_top10_min5_books.insert(
    0,
    "rank",
    range(1, len(author_quote_gap_top10_min5_books) + 1),
)

quote_author_table_dir = (
    TABLE_DIR
)
quote_author_table_dir.mkdir(parents=True, exist_ok=True)
author_quote_gap_summary.to_csv(
    quote_author_table_dir
    / "quote_author_narration_quotation_interiority_gap_summary.csv",
    index=False,
)
author_quote_gap_top10_min5_books.to_csv(
    quote_author_table_dir
    / "quote_author_narration_quotation_interiority_gap_top10_min5_books.csv",
    index=False,
)

display(
    author_quote_gap_top10_min5_books[
        [
            "rank",
            "author_display",
            "n_books",
            "n_books_with_gap",
            "gap_book_coverage",
            "mean_narration_quotation_interiority_gap",
        ]
    ].style.format(
        {
            "gap_book_coverage": "{:.1%}",
            "mean_narration_quotation_interiority_gap": "{:.3f}",
        }
    )
)

display(
    Markdown(
        "The five-book threshold is based on each author's total contribution to the current dataset. "
        "`n_books_with_gap` reports how many of those books have a nonmissing finalized gap; "
        "the author mean gives each valid book equal weight."
    )
)


### Additional publication-ready book-level comparisons

These supporting figures use exactly one observation per book. Books are never weighted by passage counts.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

PUBLICATION_OUTPUT_DIR = RESULTS_DIR
PUBLICATION_FIGURE_DIR = FIGURE_DIR
PUBLICATION_TABLE_DIR = TABLE_DIR
PUBLICATION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PUBLICATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

BROAD_GROUP_COLORS_PUBLICATION = BROAD_GROUP_COLORS.copy()

publication_book_data = book_summary_analysis.copy()
if publication_book_data["gutenberg_id"].duplicated().any():
    raise ValueError("Publication figures require exactly one row per book.")

publication_book_data["average_interiority_score"] = pd.to_numeric(
    publication_book_data["avg_interiority_score"], errors="coerce"
)
publication_book_data["publication_period"] = pd.cut(
    pd.to_numeric(publication_book_data["publication_year"], errors="coerce"),
    bins=[float("-inf"), 1840, 1900, float("inf")],
    labels=PUBLICATION_PERIODS,
    right=False,
    ordered=True,
)


publication_inventory = pd.DataFrame(
    {
        "books": [publication_book_data["gutenberg_id"].nunique()],
        "books_with_valid_score": [
            publication_book_data["average_interiority_score"].notna().sum()
        ],
        "bootstrap_repetitions": [BOOTSTRAP_REPETITIONS],
        "bootstrap_seed": [BOOTSTRAP_SEED],
    }
)
display(publication_inventory)


#### Corpus composition overview


In [ ]:
composition_data = publication_book_data.assign(
    broad_group_publication=publication_book_data["broad_group"].fillna("missing")
)
corpus_composition_summary = (
    composition_data.groupby("broad_group_publication", dropna=False)
    .agg(corpus_books=("gutenberg_id", "nunique"))
    .reset_index()
    .rename(columns={"broad_group_publication": "broad_group"})
)
corpus_composition_summary["group_label"] = corpus_composition_summary[
    "broad_group"
].map(BROAD_GROUP_LABELS)
corpus_composition_summary["corpus_pct"] = (
    corpus_composition_summary["corpus_books"]
    / publication_book_data["gutenberg_id"].nunique()
)
corpus_composition_summary = corpus_composition_summary.sort_values(
    "corpus_books", ascending=False
).reset_index(drop=True)
corpus_composition_summary.to_csv(
    PUBLICATION_TABLE_DIR / "figure1_corpus_composition.csv", index=False
)
display(
    corpus_composition_summary.style.format(
        {"corpus_pct": "{:.1%}"}
    )
)

plot_composition = corpus_composition_summary.copy()
y = np.arange(len(plot_composition))
fig, ax = plt.subplots(figsize=(9.2, 5.2))
bars = ax.barh(
    y,
    plot_composition["corpus_books"],
    color=[
        BROAD_GROUP_COLORS_PUBLICATION.get(group, RESIDUAL_GRAY)
        for group in plot_composition["broad_group"]
    ],
    height=0.68,
)
ax.set_yticks(y, plot_composition["group_label"])
ax.invert_yaxis()
ax.set_xlabel("Number of books")
ax.set_ylabel("")
ax.set_title("Corpus composition by broad group", loc="left")
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#E4E7EA", linewidth=0.8)
largest_count = max(plot_composition["corpus_books"])
ax.set_xlim(0, largest_count * 1.28)
for bar, row in zip(bars, plot_composition.itertuples()):
    ax.text(
        bar.get_width() + largest_count * 0.018,
        bar.get_y() + bar.get_height() / 2,
        f"{row.corpus_books:,} ({row.corpus_pct:.1%})",
        va="center",
        fontsize=9,
        color="#343A40",
    )
sns.despine(ax=ax)
plt.tight_layout()
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure1_corpus_composition")
plt.show()

largest_group = plot_composition.iloc[0]
display(
    Markdown(
        f"**Brief interpretation.** {largest_group['group_label']} is the largest mutually exclusive corpus group "
        f"({largest_group['corpus_books']:,} books; {largest_group['corpus_pct']:.1%}). "
        "This figure reports corpus composition only; interiority estimates are shown separately below."
    )
)


#### Broad-group interiority summary


In [ ]:
broad_group_summary_rows = []
composition_with_scores = publication_book_data.assign(
    broad_group_publication=publication_book_data["broad_group"].fillna("missing")
)
present_broad_groups = set(composition_with_scores["broad_group_publication"])
for broad_group in BROAD_GROUP_ORDER:
    if broad_group not in present_broad_groups:
        continue
    group_scores = composition_with_scores.loc[
        composition_with_scores["broad_group_publication"].eq(broad_group),
        "average_interiority_score",
    ]
    summary = bootstrap_mean_summary(
        group_scores,
        "broad_group_interiority",
        broad_group,
    )
    broad_group_summary_rows.append(
        {
            "broad_group": broad_group,
            "group_label": BROAD_GROUP_LABELS[broad_group],
            **summary,
        }
    )

broad_group_interiority_bootstrap = pd.DataFrame(broad_group_summary_rows)
broad_group_interiority_bootstrap.to_csv(
    PUBLICATION_TABLE_DIR / "figure1b_broad_group_interiority_bootstrap.csv",
    index=False,
)
display(
    broad_group_interiority_bootstrap.style.format(
        {
            "estimate": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(9.2, 5.2))
y = np.arange(len(broad_group_interiority_bootstrap))
for position, row in enumerate(broad_group_interiority_bootstrap.itertuples()):
    color = BROAD_GROUP_COLORS_PUBLICATION.get(row.broad_group, RESIDUAL_GRAY)
    is_domestic_only = row.broad_group == "domestic_only"
    is_domestic_hybrid = row.broad_group == "domestic_and_genre"
    marker = "o" if is_domestic_only else ("D" if is_domestic_hybrid else "s")
    linewidth = 2.5 if is_domestic_only else (2.0 if is_domestic_hybrid else 1.5)
    marker_size = 8 if is_domestic_only else 7
    marker_face = color if row.sample_status == "standard_n_ge_20" else "white"

    if row.ci_displayed:
        ax.hlines(
            position,
            row.ci_low,
            row.ci_high,
            color=color,
            linewidth=linewidth,
        )
        ax.plot(
            row.estimate,
            position,
            marker=marker,
            markersize=marker_size,
            markerfacecolor=marker_face,
            markeredgecolor=color,
            markeredgewidth=1.6,
            linestyle="none",
        )
    else:
        ax.plot(
            row.estimate,
            position,
            marker="x",
            markersize=8,
            color=color,
            markeredgewidth=1.8,
            linestyle="none",
        )
    ax.text(
        row.estimate + 0.012,
        position - 0.16,
        f"n={row.n_books}",
        fontsize=8,
        color="#555B61",
    )

ax.set_yticks(y, broad_group_interiority_bootstrap["group_label"])
ax.invert_yaxis()
ax.set_xlabel("Mean book-level interiority score (95% bootstrap CI)")
ax.set_ylabel("")
ax.set_title("Interiority by mutually exclusive broad group", loc="left")
finite_bounds = broad_group_interiority_bootstrap[["ci_low", "ci_high"]].to_numpy(float)
bound_min = np.nanmin(finite_bounds)
bound_max = np.nanmax(finite_bounds)
ax.set_xlim(bound_min - 0.08, bound_max + 0.12)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#E4E7EA", linewidth=0.8)
sns.despine(ax=ax)
plt.tight_layout()
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure1b_broad_group_interiority_summary")
plt.show()

highest_broad_group = broad_group_interiority_bootstrap.loc[
    broad_group_interiority_bootstrap["estimate"].idxmax()
]
display(
    Markdown(
        f"**Brief interpretation.** {highest_broad_group['group_label']} has the highest descriptive "
        f"book-level mean ({highest_broad_group['estimate']:.2f}). The groups are mutually exclusive; "
        "intervals summarize book-level bootstrap variability rather than passage-level uncertainty."
    )
)


#### Overall genre comparison


In [ ]:
overall_genre_rows = []
for group_column in GENRE_COLUMNS:
    group_scores = publication_book_data.loc[
        publication_book_data[group_column].eq(1),
        "average_interiority_score",
    ]
    summary = bootstrap_mean_summary(
        group_scores,
        "overall_genre",
        group_column,
    )
    overall_genre_rows.append(
        {
            "group_column": group_column,
            "group_label": GENRE_LABELS[group_column],
            **summary,
        }
    )

overall_genre_bootstrap = pd.DataFrame(overall_genre_rows)
domestic_row = overall_genre_bootstrap.loc[
    overall_genre_bootstrap["group_column"].eq("is_domestic")
]
other_rows = overall_genre_bootstrap.loc[
    ~overall_genre_bootstrap["group_column"].eq("is_domestic")
].sort_values("estimate", ascending=False)
overall_genre_bootstrap = pd.concat(
    [domestic_row, other_rows], ignore_index=True
)
overall_genre_bootstrap.to_csv(
    PUBLICATION_TABLE_DIR / "figure2_overall_genre_bootstrap.csv",
    index=False,
)
display(
    overall_genre_bootstrap.style.format(
        {
            "estimate": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(9.2, 6.0))
y = np.arange(len(overall_genre_bootstrap))
for position, row in enumerate(overall_genre_bootstrap.itertuples()):
    is_domestic = row.group_column == "is_domestic"
    color = DOMESTIC_ORANGE if is_domestic else NONDOMESTIC_BLUE
    marker = "o" if is_domestic else "s"
    linewidth = 2.5 if is_domestic else 1.5
    marker_face = color if row.sample_status == "standard_n_ge_20" else "white"

    if row.ci_displayed:
        ax.hlines(position, row.ci_low, row.ci_high, color=color, linewidth=linewidth)
        ax.plot(
            row.estimate,
            position,
            marker=marker,
            markersize=8 if is_domestic else 7,
            markerfacecolor=marker_face,
            markeredgecolor=color,
            markeredgewidth=1.6,
            linestyle="none",
        )
    else:
        ax.plot(
            row.estimate,
            position,
            marker="x",
            markersize=8,
            color=color,
            markeredgewidth=1.8,
            linestyle="none",
        )
    ax.text(
        row.estimate + 0.012,
        position - 0.16,
        f"n={row.n_books}",
        fontsize=8,
        color="#555B61",
    )

ax.set_yticks(y, overall_genre_bootstrap["group_label"])
ax.invert_yaxis()
ax.set_xlabel("Mean book-level interiority score (95% bootstrap CI)")
ax.set_ylabel("")
ax.set_title("Overall interiority by overlapping corpus indicator", loc="left")
finite_bounds = overall_genre_bootstrap[["ci_low", "ci_high"]].to_numpy(float)
bound_min = np.nanmin(finite_bounds)
bound_max = np.nanmax(finite_bounds)
ax.set_xlim(bound_min - 0.08, bound_max + 0.12)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#E4E7EA", linewidth=0.8)
sns.despine(ax=ax)
fig.text(
    0.01,
    -0.01,
    "Groups overlap: a book may contribute to multiple estimates; sample sizes must not be summed.",
    fontsize=8.5,
    color="#555B61",
)
plt.tight_layout()
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure2_overall_genre_comparison")
plt.show()

highest_overall = overall_genre_bootstrap.loc[
    overall_genre_bootstrap["estimate"].idxmax()
]
lowest_overall = overall_genre_bootstrap.loc[
    overall_genre_bootstrap["estimate"].idxmin()
]
display(
    Markdown(
        f"**Brief interpretation.** The highest descriptive mean is for {highest_overall['group_label']} "
        f"({highest_overall['estimate']:.2f}); the lowest is for {lowest_overall['group_label']} "
        f"({lowest_overall['estimate']:.2f}). Groups overlap, so these are not mutually exclusive corpus partitions."
    )
)


#### Genre × period small multiples


In [ ]:
genre_period_rows = []
comparison_groups = ["is_domestic", *SUBSTANTIVE_GENRES]
for group_column in comparison_groups:
    for period in PUBLICATION_PERIODS:
        cell_scores = publication_book_data.loc[
            publication_book_data[group_column].eq(1)
            & publication_book_data["publication_period"].eq(period),
            "average_interiority_score",
        ]
        summary = bootstrap_mean_summary(
            cell_scores,
            "genre_period",
            group_column,
            period,
        )
        genre_period_rows.append(
            {
                "group_column": group_column,
                "group_label": GENRE_LABELS[group_column],
                "publication_period": period,
                **summary,
            }
        )

genre_period_bootstrap = pd.DataFrame(genre_period_rows)
genre_period_bootstrap["publication_period"] = pd.Categorical(
    genre_period_bootstrap["publication_period"],
    categories=PUBLICATION_PERIODS,
    ordered=True,
)
genre_period_bootstrap = genre_period_bootstrap.sort_values(
    ["group_column", "publication_period"]
)
genre_period_bootstrap.to_csv(
    PUBLICATION_TABLE_DIR / "figure3_genre_period_bootstrap.csv",
    index=False,
)
genre_period_cell_counts = (
    genre_period_bootstrap.pivot(
        index="group_label",
        columns="publication_period",
        values="n_books",
    )
    .reindex(
        index=[GENRE_LABELS[column] for column in comparison_groups],
        columns=PUBLICATION_PERIODS,
    )
    .fillna(0)
    .astype(int)
)
genre_period_cell_counts.to_csv(
    PUBLICATION_TABLE_DIR / "figure3_genre_period_cell_counts.csv"
)
display(Markdown("**Valid book counts in every group × period cell**"))
display(genre_period_cell_counts)

finite_y = genre_period_bootstrap[
    ["estimate", "ci_low", "ci_high"]
].to_numpy(float)
y_min = np.floor((np.nanmin(finite_y) - 0.08) * 10) / 10
y_max = np.ceil((np.nanmax(finite_y) + 0.08) * 10) / 10


fig, axes = plt.subplots(2, 3, figsize=(14.5, 8.2), sharex=True, sharey=True)
period_x = np.arange(len(PUBLICATION_PERIODS))
for ax, genre_column in zip(axes.flat[:5], SUBSTANTIVE_GENRES):
    domestic_series = genre_period_bootstrap.loc[
        genre_period_bootstrap["group_column"].eq("is_domestic")
    ].set_index("publication_period").reindex(PUBLICATION_PERIODS)
    genre_series = genre_period_bootstrap.loc[
        genre_period_bootstrap["group_column"].eq(genre_column)
    ].set_index("publication_period").reindex(PUBLICATION_PERIODS)

    ax.plot(
        period_x,
        domestic_series["estimate"],
        color=DOMESTIC_ORANGE,
        linewidth=2.8,
        zorder=2,
    )
    ax.plot(
        period_x,
        genre_series["estimate"],
        color=NONDOMESTIC_BLUE,
        linewidth=1.7,
        zorder=2,
    )
    for x_value, (_, row) in zip(period_x, domestic_series.iterrows()):
        if pd.notna(row["estimate"]):
            draw_period_estimate(
                ax, x_value, row, DOMESTIC_ORANGE, "o", domestic=True
            )
    for x_value, (_, row) in zip(period_x, genre_series.iterrows()):
        if pd.notna(row["estimate"]):
            draw_period_estimate(
                ax, x_value, row, NONDOMESTIC_BLUE, "s", domestic=False
            )

    last_domestic = domestic_series["estimate"].dropna()
    last_genre = genre_series["estimate"].dropna()
    if len(last_domestic):
        ax.text(
            2.08,
            last_domestic.iloc[-1] + 0.015,
            "Domestic",
            color=DOMESTIC_ORANGE,
            fontsize=8,
            fontweight="bold",
        )
    if len(last_genre):
        ax.text(
            2.08,
            last_genre.iloc[-1] - 0.025,
            GENRE_LABELS[genre_column],
            color=NONDOMESTIC_BLUE,
            fontsize=8,
        )
    ax.set_title(GENRE_LABELS[genre_column], loc="left", fontsize=11)
    ax.set_xticks(period_x, PUBLICATION_PERIODS)
    ax.set_xlim(-0.2, 2.55)
    ax.set_ylim(y_min, y_max)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", color="#E4E7EA", linewidth=0.8)
    sns.despine(ax=ax)

axes.flat[5].axis("off")
status_handles = [
    Line2D([0], [0], color=DOMESTIC_ORANGE, marker="o", linewidth=2.8, label="Domestic"),
    Line2D([0], [0], color=NONDOMESTIC_BLUE, marker="s", linewidth=1.7, label="Panel genre"),
    Line2D([0], [0], color=NONDOMESTIC_DARK, marker="o", markerfacecolor="white", linewidth=1.2, label="Limited: 10–19 books"),
    Line2D([0], [0], color=NONDOMESTIC_DARK, marker="x", linewidth=0, label="Mean only: <10 books"),
]
axes.flat[5].legend(
    handles=status_handles,
    frameon=False,
    loc="center left",
    title="Encoding",
)
fig.supylabel("Mean book-level interiority score")
fig.suptitle("Domestic and genre interiority across publication periods", x=0.04, ha="left")
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure3_genre_period_small_multiples")
plt.show()

sparse_cells = genre_period_bootstrap.loc[
    genre_period_bootstrap["n_books"].lt(10)
]
display(
    Markdown(
        f"**Brief interpretation.** Period patterns vary across genres. The saved count table identifies "
        f"{len(sparse_cells)} cell(s) with fewer than 10 valid books; those cells show a descriptive mean only or no estimate, without a displayed CI."
    )
)


#### Genre difference from Domestic heatmap


In [ ]:
difference_rows = []
for genre_column in SUBSTANTIVE_GENRES:
    for period in PUBLICATION_PERIODS:
        genre_scores = publication_book_data.loc[
            publication_book_data[genre_column].eq(1)
            & publication_book_data["publication_period"].eq(period),
            "average_interiority_score",
        ]
        domestic_scores = publication_book_data.loc[
            publication_book_data["is_domestic"].eq(1)
            & publication_book_data["publication_period"].eq(period),
            "average_interiority_score",
        ]
        summary = bootstrap_difference_summary(
            genre_scores,
            domestic_scores,
            "genre_minus_domestic",
            genre_column,
            period,
        )
        difference_rows.append(
            {
                "genre_column": genre_column,
                "genre_label": GENRE_LABELS[genre_column],
                "publication_period": period,
                **summary,
            }
        )

genre_domestic_difference_bootstrap = pd.DataFrame(difference_rows)
genre_domestic_difference_bootstrap["publication_period"] = pd.Categorical(
    genre_domestic_difference_bootstrap["publication_period"],
    categories=PUBLICATION_PERIODS,
    ordered=True,
)
genre_domestic_difference_bootstrap = genre_domestic_difference_bootstrap.sort_values(
    ["genre_column", "publication_period"]
)
genre_domestic_difference_bootstrap.to_csv(
    PUBLICATION_TABLE_DIR / "figure4_5_genre_minus_domestic_bootstrap.csv",
    index=False,
)
display(
    genre_domestic_difference_bootstrap.style.format(
        {
            "genre_mean": "{:.3f}",
            "domestic_mean": "{:.3f}",
            "difference": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
        }
    )
)

genre_order = [GENRE_LABELS[column] for column in SUBSTANTIVE_GENRES]
difference_matrix = (
    genre_domestic_difference_bootstrap.pivot(
        index="genre_label",
        columns="publication_period",
        values="difference",
    )
    .reindex(index=genre_order, columns=PUBLICATION_PERIODS)
)
sparse_matrix = (
    genre_domestic_difference_bootstrap.assign(
        sparse=lambda frame: frame[["n_genre", "n_domestic"]].min(axis=1).lt(10)
    )
    .pivot(
        index="genre_label",
        columns="publication_period",
        values="sparse",
    )
    .reindex(index=genre_order, columns=PUBLICATION_PERIODS)
    .fillna(True)
)
finite_differences = difference_matrix.to_numpy(float)
max_abs_difference = np.nanmax(np.abs(finite_differences))
heatmap_limit = max(0.1, np.ceil(max_abs_difference * 10) / 10)

fig, ax = plt.subplots(figsize=(8.6, 5.1))
sns.heatmap(
    difference_matrix,
    cmap="vlag",
    center=0,
    vmin=-heatmap_limit,
    vmax=heatmap_limit,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Genre mean − Domestic mean"},
    annot=False,
    ax=ax,
)
for row_index, genre_label in enumerate(genre_order):
    for column_index, period in enumerate(PUBLICATION_PERIODS):
        value = difference_matrix.loc[genre_label, period]
        sparse = bool(sparse_matrix.loc[genre_label, period])
        label = "NA†" if pd.isna(value) else f"{value:+.2f}{'†' if sparse else ''}"
        ax.text(
            column_index + 0.5,
            row_index + 0.5,
            label,
            ha="center",
            va="center",
            fontsize=10,
            color="#202428",
            fontweight="bold" if sparse else "normal",
        )
        if sparse:
            ax.add_patch(
                Rectangle(
                    (column_index, row_index),
                    1,
                    1,
                    fill=False,
                    hatch="///",
                    edgecolor="#777777",
                    linewidth=0,
                )
            )
ax.set_title("Genre difference from Domestic by publication period", loc="left")
ax.set_xlabel("")
ax.set_ylabel("")
fig.text(
    0.01,
    -0.02,
    "Positive values indicate higher genre means. † Fewer than 10 books in at least one comparison group; CI not displayed.",
    fontsize=8.5,
    color="#555B61",
)
plt.tight_layout()
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure4_genre_minus_domestic_heatmap")
plt.show()

finite_difference_rows = genre_domestic_difference_bootstrap.dropna(
    subset=["difference"]
)
positive_cells = int(finite_difference_rows["difference"].gt(0).sum())
negative_cells = int(finite_difference_rows["difference"].lt(0).sum())
sparse_comparisons = int(
    genre_domestic_difference_bootstrap[["n_genre", "n_domestic"]]
    .min(axis=1)
    .lt(10)
    .sum()
)
display(
    Markdown(
        f"**Brief interpretation.** Among estimable cells, {positive_cells} genre-period means are above "
        f"Domestic and {negative_cells} are below. {sparse_comparisons} comparison(s) have fewer than 10 books "
        "in at least one group and are marked as descriptive only."
    )
)


#### Mean-difference dot plot


In [ ]:
period_offsets = {
    "Pre-1840": -0.22,
    "1840–1899": 0.0,
    "1900+": 0.22,
}

fig, ax = plt.subplots(figsize=(10.2, 6.4))
y_base = {genre: index for index, genre in enumerate(genre_order)}
for row in genre_domestic_difference_bootstrap.itertuples():
    y_value = y_base[row.genre_label] + period_offsets[str(row.publication_period)]
    period = str(row.publication_period)
    color = PERIOD_COLORS[period]
    marker = PERIOD_MARKERS[period]

    if pd.isna(row.difference):
        ax.text(
            0,
            y_value,
            "NA",
            ha="center",
            va="center",
            fontsize=8,
            color=RESIDUAL_GRAY,
        )
    elif row.sample_status == "standard_n_ge_20":
        ax.hlines(y_value, row.ci_low, row.ci_high, color=color, linewidth=1.5)
        ax.plot(
            row.difference,
            y_value,
            marker=marker,
            markersize=7.5,
            markerfacecolor=color,
            markeredgecolor=color,
            linestyle="none",
        )
    elif row.sample_status == "limited_n_10_19":
        ax.hlines(y_value, row.ci_low, row.ci_high, color=color, linewidth=1.5)
        ax.plot(
            row.difference,
            y_value,
            marker=marker,
            markersize=7.5,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=1.7,
            linestyle="none",
        )
    else:
        ax.plot(
            row.difference,
            y_value,
            marker="x",
            markersize=8,
            color=color,
            markeredgewidth=1.8,
            linestyle="none",
        )

ax.axvline(0, color="#555B61", linewidth=1.1)
ax.set_yticks(range(len(genre_order)), genre_order)
ax.invert_yaxis()
ax.set_xlabel("Mean difference: genre − Domestic (95% bootstrap CI)")
ax.set_ylabel("")
ax.set_title("Genre differences from Domestic across publication periods", loc="left")
finite_ci = genre_domestic_difference_bootstrap[
    ["difference", "ci_low", "ci_high"]
].to_numpy(float)
difference_limit = np.nanmax(np.abs(finite_ci))
x_limit = np.ceil((difference_limit + 0.04) * 10) / 10
ax.set_xlim(-x_limit, x_limit)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#E4E7EA", linewidth=0.8)
sns.despine(ax=ax)

period_handles = [
    Line2D(
        [0], [0],
        color=PERIOD_COLORS[period],
        marker=PERIOD_MARKERS[period],
        linewidth=1.5,
        label=period,
    )
    for period in PUBLICATION_PERIODS
]
ax.legend(
    handles=period_handles,
    title="Publication period",
    frameon=False,
    loc="lower right",
)
fig.text(
    0.01,
    -0.01,
    "Filled marker: n≥20 in both groups; open marker: 10–19 in at least one group; ×: mean only (<10); NA: no genre estimate.",
    fontsize=8.5,
    color="#555B61",
)
plt.tight_layout()
save_figure(fig, PUBLICATION_FIGURE_DIR, "figure5_genre_minus_domestic_dotplot")
plt.show()

displayed_intervals = genre_domestic_difference_bootstrap.loc[
    genre_domestic_difference_bootstrap["ci_displayed"]
]
intervals_crossing_zero = int(
    (
        displayed_intervals["ci_low"].le(0)
        & displayed_intervals["ci_high"].ge(0)
    ).sum()
)
display(
    Markdown(
        f"**Brief interpretation.** The plot emphasizes direction and bootstrap interval width. "
        f"{intervals_crossing_zero} displayed interval(s) span zero; this is a descriptive summary of "
        "book-level bootstrap variability, not a causal or formal significance claim."
    )
)


#### Saved publication outputs

Generated figures are saved as 300-dpi PNG files and vector PDF files in `results/figures/`. Supporting descriptive tables, including summarized plotting datasets, bootstrap bounds, cell sample sizes, and display-status fields, are saved in `results/tables/`.
